In [97]:
!pip install bs4

In [98]:
!pip install lxml

In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.tokenize import sent_tokenize,word_tokenize
from numba.cpython.setobj import set_ne
from sklearn.model_selection import train_test_split
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from nltk.corpus import stopwords
import gensim
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

from bs4 import BeautifulSoup
from sklearn.metrics import classification_report,accuracy_score


In [3]:
dataset=pd.read_csv('../dataset/kindle_review.csv')

In [4]:
dataset

,index,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000
...,...,...,...,...,...,...,...,...,...,...,...
11995,11995,2183,B001DUGORO,"[0, 0]",4,Valentine cupid is a vampire- Jena and Ian ano...,"02 28, 2014",A1OKS5Q1HD8WQC,lisa jon jung,jena,1393545600
11996,11996,6272,B002JCSFSQ,"[2, 2]",5,I have read all seven books in this series. Ap...,"05 16, 2011",AQRSPXLNEQAMA,TerryLP,Peacekeepers Series,1305504000
11997,11997,12483,B0035N1V7K,"[0, 1]",3,This book really just wasn't my cuppa. The si...,"07 26, 2013",A2T5QLT5VXOJAK,hwilson,a little creepy,1374796800
11998,11998,3640,B001W1XT40,"[1, 2]",1,"tried to use it to charge my kindle, it didn't...","09 17, 2013",A28MHD2DDY6DXB,"Allison A. Slater ""Gryphon50""",didn't work,1379376000


In [5]:
dataset.columns

Index(['index', 'Unnamed: 0', 'asin', 'helpful', 'rating', 'reviewText',
       'reviewTime', 'reviewerID', 'reviewerName', 'summary',
       'unixReviewTime'],
      dtype='object')

In [6]:
dataset=dataset.drop(columns=['index', 'Unnamed: 0', 'asin','reviewTime', 'reviewerID','unixReviewTime','reviewerName','helpful','summary'],axis=1)

In [7]:
dataset

,rating,reviewText
0,3,"Jace Rankin may be short, but he's nothing to ..."
1,5,Great short read. I didn't want to put it dow...
2,3,I'll start by saying this is the first of four...
3,3,Aggie is Angela Lansbury who carries pocketboo...
4,4,I did not expect this type of book to be in li...
...,...,...
11995,4,Valentine cupid is a vampire- Jena and Ian ano...
11996,5,I have read all seven books in this series. Ap...
11997,3,This book really just wasn't my cuppa. The si...
11998,1,"tried to use it to charge my kindle, it didn't..."


In [8]:
dataset['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [9]:
def fixRating(rating):
    if rating <=3:
        return 0
    else:
        return 1

In [10]:
dataset['sentiment']=dataset['rating'].apply(fixRating)

In [11]:
dataset

,rating,reviewText,sentiment
0,3,"Jace Rankin may be short, but he's nothing to ...",0
1,5,Great short read. I didn't want to put it dow...,1
2,3,I'll start by saying this is the first of four...,0
3,3,Aggie is Angela Lansbury who carries pocketboo...,0
4,4,I did not expect this type of book to be in li...,1
...,...,...,...
11995,4,Valentine cupid is a vampire- Jena and Ian ano...,1
11996,5,I have read all seven books in this series. Ap...,1
11997,3,This book really just wasn't my cuppa. The si...,0
11998,1,"tried to use it to charge my kindle, it didn't...",0


In [12]:
dataset['sentiment'].value_counts()

sentiment
0    6000
1    6000
Name: count, dtype: int64

In [13]:
# Dataset is fixed Now we need to preprocess the text
sentences=[]

for elements in dataset['reviewText']:
    sent=sent_tokenize(elements.lower())
    sentences.append(sent)

In [14]:
sentences

[["jace rankin may be short, but he's nothing to mess with, as the man who was just hauled out of the saloon by the undertaker knows now.",
  "he's a famous bounty hunter in oregon in the 1890s who, when he shot the man in the saloon, just finished a years long quest to avenge his sister's murder and is now trying to figure out what to do next.",
  'when the snotty-nosed farm boy he just rescued from a gang of bullies offers him money to kill a man who forced him off his ranch, he reluctantly agrees to bring the man to justice, but not to kill him outright.',
  'but, first he needs to tell his sister\'s widower the news.kyla "kyle" springer bailey has been riding the trails and sleeping on the ground for the past month while trying to find jace.',
  "she wants revenge on the man who killed her husband and took her ranch, amongst other crimes, and she's not so keen on the detour jace wants to take.",
  "but she realizes she's out of options, so she hides behind her boy persona as best s

In [15]:
sentences[0][1]

"he's a famous bounty hunter in oregon in the 1890s who, when he shot the man in the saloon, just finished a years long quest to avenge his sister's murder and is now trying to figure out what to do next."

In [16]:
lemmatizer=WordNetLemmatizer()

In [17]:
len(dataset)

12000

In [18]:
stop_words = set(stopwords.words('english'))

corpus = []

for text in dataset['reviewText']:

    word = re.sub('[^a-zA-Z]', ' ', str(text))
    word = re.sub(r'(http|https|ftp|ssh)://\S+', '', word)
    word = BeautifulSoup(word, 'html.parser').get_text()
    word = word.lower()
    word = word_tokenize(word)

    words = [lemmatizer.lemmatize(w) for w in word if w not in stop_words]

    corpus.append(' '.join(words))

In [19]:
corpus

['jace rankin may short nothing mess man hauled saloon undertaker know famous bounty hunter oregon shot man saloon finished year long quest avenge sister murder trying figure next snotty nosed farm boy rescued gang bully offer money kill man forced ranch reluctantly agrees bring man justice kill outright first need tell sister widower news kyla kyle springer bailey riding trail sleeping ground past month trying find jace want revenge man killed husband took ranch amongst crime keen detour jace want take realizes option hide behind boy persona best try keep pace confrontation along way get shot jace discovers kyle kyla come clean whole reason need scoundrel dead hope still help book share touching moment slow blooming romance kyla find good reason fear men hide behind boy persona watching jace slowly pull shell help conquer fear endearing pain real deeply rooted disappear face sexiness neither understandable aversion marriage magically disappear round nookie would man drifted town town 

In [20]:
len(corpus)

12000

In [21]:
X_train,X_test,y_train,y_test=train_test_split(
    corpus,dataset['sentiment'],test_size=0.20,random_state=42
)

In [22]:
X_train

['looking forward book came double space every paragraph kindle edition since action move around formatting make story hard follow die hard like want bother sad thing good book spoiled formatting fault author story good book energy read also emailed author',
 'already owned book spouse forgot already part library book unfortunate amazon safeguard place notify book already library',
 'cool forgot request rate came make mine unreliable rating doubt change rotation planet enough measure',
 'short short story basically scene party one night though admittedly entire book taken period one pure erotica hea real relationship depth sexual description really disappointed get used kindle notation find lack page number confusing really know getting load found regency romance really idea men love look took min read wish could return even felt ripped',
 'secret service agent secrests even longer service meet argus ward former secret service agent run protection agency catering rich famous secret fun

In [23]:
y_train

9182     1
11091    0
6428     1
288      0
2626     1
        ..
11964    0
5191     1
5390     0
860      0
7270     1
Name: sentiment, Length: 9600, dtype: int64

In [24]:
len(X_train)

9600

In [25]:
# Applying Bow,TfIdf, Word2Vec

bow=CountVectorizer()
tfidf=TfidfVectorizer()

In [26]:
model=Word2Vec(sentences=corpus)

In [27]:
X_train_bow=bow.fit_transform(X_train)
X_test_bow=bow.transform(X_test)

In [28]:
X_train_tfidf=tfidf.fit_transform(X_train)
X_test_tfidf=tfidf.transform(X_test)

In [29]:
tokenized_corpus = [doc.split() for doc in corpus]

In [30]:
tokenized_corpus

[['jace',
  'rankin',
  'may',
  'short',
  'nothing',
  'mess',
  'man',
  'hauled',
  'saloon',
  'undertaker',
  'know',
  'famous',
  'bounty',
  'hunter',
  'oregon',
  'shot',
  'man',
  'saloon',
  'finished',
  'year',
  'long',
  'quest',
  'avenge',
  'sister',
  'murder',
  'trying',
  'figure',
  'next',
  'snotty',
  'nosed',
  'farm',
  'boy',
  'rescued',
  'gang',
  'bully',
  'offer',
  'money',
  'kill',
  'man',
  'forced',
  'ranch',
  'reluctantly',
  'agrees',
  'bring',
  'man',
  'justice',
  'kill',
  'outright',
  'first',
  'need',
  'tell',
  'sister',
  'widower',
  'news',
  'kyla',
  'kyle',
  'springer',
  'bailey',
  'riding',
  'trail',
  'sleeping',
  'ground',
  'past',
  'month',
  'trying',
  'find',
  'jace',
  'want',
  'revenge',
  'man',
  'killed',
  'husband',
  'took',
  'ranch',
  'amongst',
  'crime',
  'keen',
  'detour',
  'jace',
  'want',
  'take',
  'realizes',
  'option',
  'hide',
  'behind',
  'boy',
  'persona',
  'best',
  'try',

In [31]:
def document_vector(doc):
    words = doc.split()
    word_vectors = []

    for word in words:
        if word in model.wv:
            word_vectors.append(model.wv[word])

    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(word_vectors, axis=0)

In [32]:
X_train_avgword2vec = np.array([document_vector(doc) for doc in X_train])
X_test_avgword2vec = np.array([document_vector(doc) for doc in X_test])

In [33]:
X_test_avgword2vec

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(2400, 100))

In [34]:
from tqdm.auto import tqdm


def train_model(model, X_train, y_train, X_test, y_test, params):

    print(f'\n---------------------------{model.__class__.__name__}---------------------------')

    cv = GridSearchCV(
        estimator=model,
        param_grid=params,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1   # shows internal CV progress
    )

    cv.fit(X_train, y_train)

    y_predicted = cv.predict(X_test)

    accuracy = accuracy_score(y_true=y_test, y_pred=y_predicted)
    cr = classification_report(y_true=y_test, y_pred=y_predicted)

    print('Best Params:', cv.best_params_)
    print('Accuracy:', accuracy)
    print(cr)




In [39]:
randomforest=RandomForestClassifier()
naive_bayes=MultinomialNB()
decisionTree=DecisionTreeClassifier()
xgb=XGBClassifier()

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'max_features': ['sqrt', 'log2']
#
}

nb_params = {
    'alpha': [0.1, 0.5, 1.0, 5.0, 10.0],
    # 'fit_prior': [True, False]
}

dt_params = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    # 'max_depth': [None, 10, 20, 30, 50],
    # 'min_samples_split': [2, 5, 10],
    # 'min_samples_leaf': [1, 2, 4],
    # 'max_features': [None, 'sqrt', 'log2']
}

xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    # 'learning_rate': [0.01, 0.1, 0.2],
    # 'subsample': [0.7, 0.8, 1.0],
    # 'colsample_bytree': [0.7, 0.8, 1.0],
    # 'gamma': [0, 0.1, 0.3]
}

In [46]:
models=[randomforest]
params=[rf_params]
trainings=[X_train_bow,X_train_tfidf]
testing=[X_test_bow,X_test_tfidf]

In [48]:
from tqdm.auto import tqdm


def train_all_model():

    total_iterations = len(models) * len(trainings)

    with tqdm(total=total_iterations, desc="Training Progress") as pbar:

        for i in range(len(models)):
            for j in range(len(trainings)):

                train_model(
                    model=models[i],
                    X_train=trainings[j],
                    y_train=y_train,
                    X_test=testing[j],
                    y_test=y_test,
                    params=params[i]
                )

                pbar.update(1)

In [49]:
train_all_model()

Training Progress:   0%|          | 0/2 [00:00<?, ?it/s]


---------------------------RandomForestClassifier---------------------------
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Params: {'max_depth': None, 'n_estimators': 300}
Accuracy: 0.8158333333333333
              precision    recall  f1-score   support

           0       0.81      0.83      0.82      1190
           1       0.82      0.81      0.82      1210

    accuracy                           0.82      2400
   macro avg       0.82      0.82      0.82      2400
weighted avg       0.82      0.82      0.82      2400


---------------------------RandomForestClassifier---------------------------
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Params: {'max_depth': None, 'n_estimators': 200}
Accuracy: 0.8058333333333333
              precision    recall  f1-score   support

           0       0.79      0.82      0.81      1190
           1       0.82      0.79      0.80      1210

    accuracy                           0.81      2400
   macro av